# 1. INE data for Andalusia

In [1]:
import logging
import pandas as pd
import sys
from pathlib import Path
from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next(
    (path for path in (working_directory, *working_directory.parents)
     if (path / "src" / "ingesta").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find the project root containing src/ingesta.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.ingesta.datos_poblacion.ine.renta import download_andalusia_income_by_municipality_from_ine
from src.ingesta.datos_poblacion.ine.padron import download_andalusia_population_by_municipality_from_ine

def preview_table(csv_path, columns=None, rows=5):
    preview = pd.read_csv(csv_path, nrows=rows, dtype=str, keep_default_na=False)
    if columns is not None:
        preview_columns = [column for column in columns if column in preview.columns]
        if not preview_columns:
            raise ValueError(f"None of the requested preview columns exist in {csv_path}.")
        preview = preview.loc[:, preview_columns]
    display(preview)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

## Andalusian population, all ages and sexes, latest five years

In [2]:
population_csv = download_andalusia_population_by_municipality_from_ine(number_of_years=5)

2026-10-19 09:07:30,077 INFO src.ingesta.datos_poblacion.ine.common: Downloaded INE table 68542.
2026-10-19 09:08:40,311 INFO src.ingesta.datos_poblacion.ine.padron: Saved 1201050 Andalusian population rows for years [2025, 2024, 2023, 2022, 2021] to C:\Users\fatima.d.villalba\OneDrive - Accenture\Documents\School of Tech\ejercicio_final\data\raw\population_andalusia.csv.


In [3]:
preview_table(population_csv)

,Provincias,Municipios,Sexo,Edad,Periodo,Total,source,downloaded_at_utc
0,04 Almería,04001 Abla,Hombres,Todas las edades,2025,660,INE,2026-10-19T07:07:30.076995+00:00
1,04 Almería,04001 Abla,Hombres,Todas las edades,2024,658,INE,2026-10-19T07:07:30.076995+00:00
2,04 Almería,04001 Abla,Hombres,Todas las edades,2023,652,INE,2026-10-19T07:07:30.076995+00:00
3,04 Almería,04001 Abla,Hombres,Todas las edades,2022,643,INE,2026-10-19T07:07:30.076995+00:00
4,04 Almería,04001 Abla,Hombres,Todas las edades,2021,648,INE,2026-10-19T07:07:30.076995+00:00


## Municipal mean net income, latest year

In [4]:
income_csv = download_andalusia_income_by_municipality_from_ine(number_of_years=1)

2026-10-19 09:09:25,321 INFO src.ingesta.datos_poblacion.ine.common: Downloaded INE table 30824.
2026-10-19 09:09:45,529 INFO src.ingesta.datos_poblacion.ine.renta: Saved 786 Andalusian income rows for years [2023] to C:\Users\fatima.d.villalba\OneDrive - Accenture\Documents\School of Tech\ejercicio_final\data\raw\income_andalusia.csv.


In [5]:
preview_table(income_csv)

,Municipios,Distritos,Secciones,Indicadores de renta media,Periodo,Total,source,downloaded_at_utc
0,04001 Abla,,,Renta neta media por persona,2023,12.411,INE,2026-10-19T07:09:25.321278+00:00
1,04002 Abrucena,,,Renta neta media por persona,2023,11.944,INE,2026-10-19T07:09:25.321278+00:00
2,04003 Adra,,,Renta neta media por persona,2023,10.156,INE,2026-10-19T07:09:25.321278+00:00
3,04004 Albanchez,,,Renta neta media por persona,2023,12.035,INE,2026-10-19T07:09:25.321278+00:00
4,04005 Alboloduy,,,Renta neta media por persona,2023,11.557,INE,2026-10-19T07:09:25.321278+00:00


# 2. OpenStreetMap gyms

In [6]:
from src.ingesta.gimnasios.osm import download_andalusia_gyms_from_osm

In [7]:
gyms_csv = download_andalusia_gyms_from_osm()

2026-10-19 09:09:45,858 INFO src.ingesta.gimnasios.osm: Downloading OSM data for Almería.
2026-10-19 09:10:46,855 INFO src.ingesta.gimnasios.osm: Downloading OSM data for Cádiz.
2026-10-19 09:11:46,281 INFO src.ingesta.gimnasios.osm: Downloading OSM data for Córdoba.
2026-10-19 09:11:46,454 WARNING src.ingesta.common.ingestion: OpenStreetMap request to https://overpass-api.de/api/interpreter failed (attempt 1/3): 502 Server Error: Bad Gateway for url: https://overpass-api.de/api/interpreter
2026-10-19 09:11:46,454 INFO src.ingesta.common.ingestion: Retrying in 30 seconds.
2026-10-19 09:13:13,396 INFO src.ingesta.gimnasios.osm: Downloading OSM data for Granada.
2026-10-19 09:13:13,575 WARNING src.ingesta.common.ingestion: OpenStreetMap request to https://overpass-api.de/api/interpreter failed (attempt 1/3): 502 Server Error: Bad Gateway for url: https://overpass-api.de/api/interpreter
2026-10-19 09:13:13,577 INFO src.ingesta.common.ingestion: Retrying in 30 seconds.
2026-10-19 09:14:49,

In [8]:
preview_table(
    gyms_csv,
    columns=[
        "osm_type", "osm_id", "province", "latitude", "longitude",
        "tag_name", "tag_leisure", "tag_sport", "source", "downloaded_at_utc",
    ],
)

,osm_type,osm_id,province,latitude,longitude,tag_name,tag_leisure,tag_sport,source,downloaded_at_utc
0,node,260812441,Almería,37.1720276,-1.8339941,Go-Karting,sports_centre,karting,OpenStreetMap,2026-10-19T07:10:36.840029+00:00
1,node,1620487480,Almería,37.3453627,-2.4325406,Pabellón,sports_centre,,OpenStreetMap,2026-10-19T07:10:36.840029+00:00
2,node,1912986759,Almería,37.212201,-1.8108268,Vera Costa,fitness_station,fitness,OpenStreetMap,2026-10-19T07:10:36.840029+00:00
3,node,2452253961,Almería,36.8380866,-2.9492787,Pabellon de deportes,sports_centre,,OpenStreetMap,2026-10-19T07:10:36.840029+00:00
4,node,4432638479,Almería,36.9699743,-2.6075333,Frontón de Huécija,sports_centre,pelota,OpenStreetMap,2026-10-19T07:10:36.840029+00:00


# 3. CNIG municipality boundaries

In [9]:
from src.ingesta.limites_municipios.cnig import download_andalusia_municipality_boundaries_from_cnig

In [10]:
municipality_boundaries_csv = download_andalusia_municipality_boundaries_from_cnig()

2026-10-19 09:20:57,341 INFO src.ingesta.limites_municipios.cnig: Downloading CNIG page at index 0.
2026-10-19 09:21:23,364 INFO src.ingesta.limites_municipios.cnig: Downloading CNIG page at index 500.
2026-10-19 09:21:43,772 INFO src.ingesta.limites_municipios.cnig: Downloading CNIG page at index 786.
2026-10-19 09:21:44,043 INFO src.ingesta.limites_municipios.cnig: Saved 786 Andalusian municipality boundaries to C:\Users\fatima.d.villalba\OneDrive - Accenture\Documents\School of Tech\ejercicio_final\data\raw\municipality_boundaries_andalusia.csv.


In [11]:
preview_table(
    municipality_boundaries_csv,
    columns=["nationalCode", "name", "geometry", "source", "downloaded_at_utc"],
)

,nationalCode,name,geometry,source,downloaded_at_utc
0,34010404001,"{""GeographicalName"": {""sourceOfName"": {""nil"": ...","{""type"": ""MultiPolygon"", ""coordinates"": [[[[-2...",IGN/CNIG BDDAE,2026-10-19T07:21:20.682217+00:00
1,34010404002,"{""GeographicalName"": {""sourceOfName"": {""nil"": ...","{""type"": ""MultiPolygon"", ""coordinates"": [[[[-2...",IGN/CNIG BDDAE,2026-10-19T07:21:20.682217+00:00
2,34010404003,"{""GeographicalName"": {""sourceOfName"": {""nil"": ...","{""type"": ""MultiPolygon"", ""coordinates"": [[[[-3...",IGN/CNIG BDDAE,2026-10-19T07:21:20.682217+00:00
3,34010404004,"{""GeographicalName"": {""sourceOfName"": {""nil"": ...","{""type"": ""MultiPolygon"", ""coordinates"": [[[[-2...",IGN/CNIG BDDAE,2026-10-19T07:21:20.682217+00:00
4,34010404005,"{""GeographicalName"": {""sourceOfName"": {""nil"": ...","{""type"": ""MultiPolygon"", ""coordinates"": [[[[-2...",IGN/CNIG BDDAE,2026-10-19T07:21:20.682217+00:00


# 4. Upload CSV outputs to S3

In [12]:
import os
from dotenv import load_dotenv
from src.ingesta.common.upload_to_s3 import upload_files_to_s3

load_dotenv(project_root / ".env", override=False)
s3_bucket = os.getenv("S3_BUCKET")
if not s3_bucket or not s3_bucket.strip():
    print("S3 upload skipped: configure S3_BUCKET in the project .env file.")
else:
    uploaded_files = upload_files_to_s3(
        [population_csv, income_csv, gyms_csv, municipality_boundaries_csv],
        bucket_name=s3_bucket,
        prefix="raw",
    )

S3 upload skipped: configure S3_BUCKET in the project .env file.
